In [ ]:
# The most recently created offline run is the completed four-checkpoint report.
offline_runs = sorted((OUTPUT_DIR / 'wandb').glob('offline-run-*'), key=lambda path: path.stat().st_mtime)
if offline_runs:
    print('Sync command when networking returns:')
    print('wandb sync', offline_runs[-1])

# Post-hoc RDT validation: steps 5k, 10k, 15k, and 20k

This notebook evaluates the four completed checkpoints on the **same fixed 256-example stratified validation subset**. It uses batch size 32, five-step diffusion sampling, Qwen zero/shuffle ablations, gripper accuracy/F1, and 32 qualitative trajectory comparisons.

W&B defaults to offline mode. Scalar JSON/CSV and qualitative PNG files are always saved locally, so network loss cannot discard the report.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO_ROOT = Path('/home/ubuntu/ThinkFlow-RDT-1B')
CHECKPOINT_STEPS = [5000, 10000, 15000, 20000]
CHECKPOINTS = [REPO_ROOT / 'output_2' / f'checkpoint-{step}' for step in CHECKPOINT_STEPS]
CONFIG = REPO_ROOT / 'configs/part3_rdt1b.yaml'
VALIDATION_MANIFEST = REPO_ROOT / 'output_2/manifests/val_manifest.jsonl'
OUTPUT_DIR = REPO_ROOT / 'output_2/posthoc_validation_5k_20k'
BATCH_SIZE = 32
VALIDATION_SAMPLES = 256
QUALITATIVE_EXAMPLES = 32
SAMPLE_VALIDATION_BATCHES = 1
WANDB_MODE = 'offline'  # Change to 'online' only when networking is reliable.
FORCE = False           # True re-runs checkpoint reports that already exist.

print('Output:', OUTPUT_DIR)

In [ ]:
# Preflight: verify checkpoint completeness and recorded training steps.
for expected_step, checkpoint in zip(CHECKPOINT_STEPS, CHECKPOINTS):
    required = [checkpoint / 'rdt_full.pt', checkpoint / 'interfaces.pt', checkpoint / 'metadata.json']
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError(missing)
    metadata = json.loads((checkpoint / 'metadata.json').read_text())
    actual_step = int(metadata['global_step'])
    if actual_step != expected_step:
        raise ValueError(f'{checkpoint}: expected {expected_step}, found {actual_step}')
    size_gib = sum(path.stat().st_size for path in required[:2]) / 1024**3
    print(f'✓ step {actual_step:>5}: {size_gib:.2f} GiB model artifact')

if not VALIDATION_MANIFEST.is_file():
    raise FileNotFoundError(VALIDATION_MANIFEST)
print('✓ fixed validation manifest:', VALIDATION_MANIFEST)

In [ ]:
# Run all requested checkpoints. Completed scalar reports are skipped unless FORCE=True.
command = [
    'uv', 'run', '--no-sync', 'python', 'scripts/validate_cached_checkpoints.py',
    '--config', str(CONFIG),
    '--validation-manifest', str(VALIDATION_MANIFEST),
    '--output-dir', str(OUTPUT_DIR),
    '--batch-size', str(BATCH_SIZE),
    '--validation-samples', str(VALIDATION_SAMPLES),
    '--sample-validation-batches', str(SAMPLE_VALIDATION_BATCHES),
    '--qualitative-examples', str(QUALITATIVE_EXAMPLES),
    '--wandb-mode', WANDB_MODE,
    '--wandb-project', 'ThinkLite B0 OXE',
    '--wandb-run-name', 'posthoc-validation-steps-5k-10k-15k-20k',
]
for checkpoint in CHECKPOINTS:
    command.extend(['--checkpoint', str(checkpoint)])
if FORCE:
    command.append('--force')

print(' '.join(command))
subprocess.run(command, cwd=REPO_ROOT, check=True)

In [ ]:
import pandas as pd

comparison_path = OUTPUT_DIR / 'checkpoint_comparison.csv'
comparison = pd.read_csv(comparison_path).sort_values('checkpoint_step')
important_metrics = [
    'checkpoint_step',
    'val/loss',
    'val/sample_mse',
    'val/sampled_native10/horizon_64/rmse',
    'val/sampled_native10/gripper_open/accuracy',
    'val/sampled_native10/gripper_open/f1',
    'val/qwen_ablation/zero/denoising_loss_delta',
    'val/qwen_ablation/shuffled/denoising_loss_delta',
    'val/qwen_ablation/zero/sample_mse_delta',
    'val/qwen_ablation/shuffled/sample_mse_delta',
]
available = [column for column in important_metrics if column in comparison.columns]
comparison[available].style.format(precision=6)

In [ ]:
import matplotlib.pyplot as plt

plots = [
    ('val/loss', 'Validation denoising loss'),
    ('val/sample_mse', 'Diffusion sample MSE'),
    ('val/sampled_native10/horizon_64/rmse', 'Native-10D horizon-64 RMSE'),
    ('val/sampled_native10/gripper_open/f1', 'Gripper-open F1'),
]
figure, axes = plt.subplots(2, 2, figsize=(12, 8))
for axis, (column, title) in zip(axes.flat, plots):
    if column not in comparison:
        axis.set_visible(False)
        continue
    axis.plot(comparison['checkpoint_step'], comparison[column], marker='o')
    axis.set_title(title)
    axis.set_xlabel('Training step')
    axis.grid(alpha=0.3)
figure.tight_layout()
plt.show()

## Local and W&B outputs

- `checkpoint_<step>_metrics.json`: complete scalar metrics for each checkpoint.
- `checkpoint_<step>_qualitative/`: 32 local observation/trajectory PNG pairs plus row metadata.
- `checkpoint_comparison.csv`: cross-checkpoint table used above.
- `summary.json`: combined machine-readable report.
- `wandb/offline-run-*`: offline W&B run, including qualitative tables.

The cell above prints the exact sync command for the latest completed offline run.